# Tutorial: Panel Methods for Causal Inference

This tutorial demonstrates **Panel Data Methods** for causal inference:

1. **DML-CRE (Binary)**: Double Machine Learning with Correlated Random Effects
2. **DML-CRE (Continuous)**: For continuous treatment (dose-response)
3. **Panel QTE**: Quantile Treatment Effects via RIF regression

**Key Concepts**:
- **Mundlak Projection**: Handle correlated unit effects by including X̄ᵢ
- **Stratified Cross-fitting**: Preserve panel structure in DML
- **Clustered SE**: Account for within-unit correlation

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import expit
import sys
sys.path.insert(0, '../../src')

from causal_inference.panel import (
    PanelData,
    dml_cre,
    dml_cre_continuous,
    panel_rif_qte,
    panel_rif_qte_band,
    panel_unconditional_qte,
)

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')

---

## Part 1: Why Panel Data Methods?

### The Problem: Correlated Unit Effects

In panel data, we observe units $i$ over time $t$. The outcome model is:

$$Y_{it} = \alpha_i + X_{it}'\beta + \theta D_{it} + \varepsilon_{it}$$

where $\alpha_i$ is the **unit fixed effect**.

**The Problem**: If $\alpha_i$ correlates with treatment $D_{it}$, standard estimators are biased.

### The Solution: Mundlak Projection

**Mundlak (1978)** showed that we can model the correlation:

$$\alpha_i = \gamma' \bar{X}_i + \eta_i$$

where $\bar{X}_i = \frac{1}{T_i} \sum_t X_{it}$ is the unit mean of covariates.

**Solution**: Include $\bar{X}_i$ as additional covariates. This "projects out" the correlation.

### Visualizing the Correlation Problem

In [ ]:
# Demonstrate correlated unit effects
n_units = 20
n_periods = 5

# Unit means of covariates
X_bar = np.random.randn(n_units)

# Unit effects CORRELATED with X_bar
alpha = 0.8 * X_bar + 0.2 * np.random.randn(n_units)

# Plot the correlation
plt.figure(figsize=(8, 5))
plt.scatter(X_bar, alpha, s=60, alpha=0.7)
plt.xlabel(r'Unit Mean $\bar{X}_i$', fontsize=12)
plt.ylabel(r'Unit Effect $\alpha_i$', fontsize=12)
plt.title(r'Correlated Unit Effects: $\alpha_i$ vs $\bar{X}_i$', fontsize=14)

# Add regression line
slope, intercept = np.polyfit(X_bar, alpha, 1)
plt.plot(X_bar, intercept + slope * X_bar, 'r--', label=f'Slope = {slope:.2f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Correlation: {np.corrcoef(X_bar, alpha)[0,1]:.3f}")
print("\n→ If we ignore this correlation, treatment effect estimates are biased!")

---

## Part 2: DML-CRE (Binary Treatment)

### Algorithm Overview

DML-CRE combines Double Machine Learning with Correlated Random Effects:

1. **Mundlak Augmentation**: Compute $\bar{X}_i$ and augment covariates: $\tilde{X}_{it} = [X_{it}, \bar{X}_i]$
2. **Stratified K-Fold**: Split by unit (not observation) to preserve panel structure
3. **Cross-fit Nuisance Models**:
   - Outcome model: $\hat{m}(X, \bar{X}) = E[Y | X, \bar{X}]$
   - Propensity: $\hat{e}(X, \bar{X}) = P(D=1 | X, \bar{X})$
4. **Residualize**: $\tilde{Y} = Y - \hat{m}$, $\tilde{D} = D - \hat{e}$
5. **Estimate ATE**: $\hat{\theta} = \frac{\sum \tilde{Y} \cdot \tilde{D}}{\sum \tilde{D}^2}$
6. **Clustered SE**: Aggregate influence function by unit

### Example: Employee Training Program

**Scenario**: A company tracks employees over quarters. Some receive training (D=1). We measure productivity.

In [ ]:
def generate_panel_dgp(
    n_units: int = 100,
    n_periods: int = 8,
    n_covariates: int = 3,
    true_ate: float = 2.0,
    unit_effect_strength: float = 0.5,
    seed: int = 42,
):
    """Generate panel data with known ATE.
    
    Model:
        αᵢ = unit_effect_strength × X̄ᵢ,₀ (correlated unit effects)
        Dᵢₜ ~ Bernoulli(logit(0.3 × Xᵢₜ,₀))
        Yᵢₜ = αᵢ + 0.5×Xᵢₜ,₀ + 0.3×Xᵢₜ,₁ + θ×Dᵢₜ + εᵢₜ
    
    Returns
    -------
    dict with: panel (PanelData), true_ate, unit_effects
    """
    rng = np.random.default_rng(seed)
    n_obs = n_units * n_periods
    
    # Unit and time indices
    unit_id = np.repeat(np.arange(n_units), n_periods)
    time = np.tile(np.arange(n_periods), n_units)
    
    # Covariates (vary within unit)
    X = rng.normal(0, 1, (n_obs, n_covariates))
    
    # Compute unit means
    X_bar = np.zeros((n_units, n_covariates))
    for i in range(n_units):
        mask = unit_id == i
        X_bar[i] = X[mask].mean(axis=0)
    
    # Unit effects (correlated with X_bar[:,0])
    alpha_unit = unit_effect_strength * X_bar[:, 0] + 0.3 * rng.normal(0, 1, n_units)
    alpha = alpha_unit[unit_id]  # Expand to observation level
    
    # Treatment (depends on current X, not unit effect directly)
    propensity = expit(0.3 * X[:, 0])
    D = rng.binomial(1, propensity)
    
    # Outcome
    epsilon = rng.normal(0, 1, n_obs)
    Y = alpha + 0.5 * X[:, 0] + 0.3 * X[:, 1] + true_ate * D + epsilon
    
    # Create PanelData
    panel = PanelData(
        outcomes=Y,
        treatment=D,
        covariates=X,
        unit_id=unit_id,
        time=time,
    )
    
    return {
        'panel': panel,
        'true_ate': true_ate,
        'unit_effects': alpha_unit,
        'X_bar': X_bar,
    }

In [ ]:
# Generate data
data = generate_panel_dgp(
    n_units=100,
    n_periods=8,
    true_ate=2.0,
    unit_effect_strength=0.5,
    seed=42,
)

panel = data['panel']
true_ate = data['true_ate']

print("=== Panel Data Summary ===")
print(f"Units: {panel.n_units}")
print(f"Periods: {panel.n_periods}")
print(f"Total obs: {panel.n_obs}")
print(f"Balanced: {panel.is_balanced}")
print(f"Covariates: {panel.n_covariates}")
print(f"\nTrue ATE: {true_ate}")
print(f"Treatment rate: {panel.treatment.mean():.1%}")

### Run DML-CRE

In [ ]:
# Estimate with DML-CRE
result = dml_cre(
    panel,
    n_folds=5,
    outcome_model='ridge',
    propensity_model='logistic',
)

print("=== DML-CRE Results ===")
print(f"ATE: {result.ate:.3f}")
print(f"SE: {result.ate_se:.3f}")
print(f"95% CI: [{result.ci_lower:.3f}, {result.ci_upper:.3f}]")
print(f"\nTrue ATE: {true_ate}")
print(f"Bias: {result.ate - true_ate:.3f}")
print(f"Covers true value: {result.ci_lower < true_ate < result.ci_upper}")

### Examine Fold Estimates

In [ ]:
# Plot fold-by-fold estimates
fold_estimates = result.fold_estimates
fold_ses = result.fold_ses

plt.figure(figsize=(10, 5))
folds = np.arange(1, len(fold_estimates) + 1)
plt.errorbar(folds, fold_estimates, yerr=1.96 * np.array(fold_ses), 
             fmt='o', capsize=5, markersize=8)
plt.axhline(y=result.ate, color='blue', linestyle='-', label=f"Overall ATE = {result.ate:.3f}")
plt.axhline(y=true_ate, color='green', linestyle='--', label=f'True ATE = {true_ate}')
plt.xlabel('Fold', fontsize=12)
plt.ylabel('ATE Estimate', fontsize=12)
plt.title('DML-CRE: Fold-by-Fold Estimates', fontsize=14)
plt.legend()
plt.xticks(folds)
plt.tight_layout()
plt.show()

print(f"Fold estimates SD: {np.std(fold_estimates):.3f}")
print("→ Stable across folds = good cross-fitting")

---

## Part 3: DML-CRE with Continuous Treatment

When treatment is continuous (e.g., drug dosage, hours of training), we use regression instead of classification for the treatment model.

### Example: Drug Dosage Study

**Scenario**: Patients visit a clinic over time. Dosage varies. We measure symptom scores.

In [ ]:
def generate_continuous_treatment_dgp(
    n_units: int = 100,
    n_periods: int = 6,
    true_effect: float = 1.5,  # Effect per unit of dosage
    seed: int = 123,
):
    """Generate panel data with continuous treatment."""
    rng = np.random.default_rng(seed)
    n_obs = n_units * n_periods
    
    unit_id = np.repeat(np.arange(n_units), n_periods)
    time = np.tile(np.arange(n_periods), n_units)
    
    # Covariates: age, severity
    X = rng.normal(0, 1, (n_obs, 2))
    
    # Unit effects (patient baseline)
    X_bar = np.zeros((n_units, 2))
    for i in range(n_units):
        X_bar[i] = X[unit_id == i].mean(axis=0)
    alpha_unit = 0.4 * X_bar[:, 0] + rng.normal(0, 0.5, n_units)
    alpha = alpha_unit[unit_id]
    
    # Continuous treatment (dosage 0-10)
    D = 5 + 0.3 * X[:, 1] + rng.normal(0, 1.5, n_obs)
    D = np.clip(D, 0, 10)
    
    # Outcome (symptom score - lower is better)
    Y = 50 + alpha + 2 * X[:, 0] - true_effect * D + rng.normal(0, 2, n_obs)
    
    panel = PanelData(
        outcomes=Y,
        treatment=D,
        covariates=X,
        unit_id=unit_id,
        time=time,
    )
    
    return {'panel': panel, 'true_effect': true_effect}

# Generate
data_cont = generate_continuous_treatment_dgp()
panel_cont = data_cont['panel']

print(f"Treatment range: [{panel_cont.treatment.min():.1f}, {panel_cont.treatment.max():.1f}]")
print(f"Treatment mean: {panel_cont.treatment.mean():.2f}")

In [ ]:
# Estimate with DML-CRE Continuous
result_cont = dml_cre_continuous(
    panel_cont,
    n_folds=5,
    outcome_model='ridge',
    treatment_model='ridge',
)

print("=== DML-CRE Continuous Results ===")
print(f"Marginal Effect: {result_cont.ate:.3f}")
print(f"SE: {result_cont.ate_se:.3f}")
print(f"95% CI: [{result_cont.ci_lower:.3f}, {result_cont.ci_upper:.3f}]")
print(f"\nTrue Effect: {data_cont['true_effect']}")
print(f"Interpretation: Each unit increase in dosage reduces symptom score by ~{abs(result_cont.ate):.2f}")

---

## Part 4: Panel Quantile Treatment Effects

The ATE tells us the **mean** effect. But what about **distributional** effects?

- Does treatment help low performers more than high performers?
- Does training raise the floor (10th percentile) or the ceiling (90th percentile)?

### RIF Regression

**Firpo, Fortin, Lemieux (2009)** introduced Recentered Influence Functions:

$$RIF(Y; q_\tau) = q_\tau + \frac{\tau - \mathbb{1}(Y \leq q_\tau)}{f_Y(q_\tau)}$$

**Key insight**: Regressing $RIF(Y; q_\tau)$ on covariates gives **unconditional quantile** effects.

In [ ]:
# Use our original binary treatment panel
# Estimate QTE at the median
qte_50 = panel_rif_qte(panel, quantile=0.5)

print("=== Panel QTE at Median ===")
print(f"QTE(0.50): {qte_50.qte:.3f}")
print(f"SE: {qte_50.qte_se:.3f}")
print(f"95% CI: [{qte_50.ci_lower:.3f}, {qte_50.ci_upper:.3f}]")
print(f"\n→ Median effect similar to mean (ATE = {result.ate:.3f})")

### Quantile Band: Effects Across Distribution

In [ ]:
# Estimate QTE at multiple quantiles
qte_band = panel_rif_qte_band(
    panel,
    quantiles=[0.10, 0.25, 0.50, 0.75, 0.90],
)

print("=== Panel QTE Band ===")
print(f"{'Quantile':>10} {'QTE':>10} {'SE':>10} {'95% CI':>20}")
print("-" * 55)
for q, qte, se, lo, hi in zip(
    qte_band.quantiles,
    qte_band.qtes,
    qte_band.qte_ses,
    qte_band.ci_lowers,
    qte_band.ci_uppers,
):
    print(f"{q:>10.0%} {qte:>10.3f} {se:>10.3f} [{lo:>7.3f}, {hi:>7.3f}]")

In [ ]:
# Visualize the quantile band
plt.figure(figsize=(10, 6))

quantiles = qte_band.quantiles
qtes = qte_band.qtes
qte_ses = qte_band.qte_ses

plt.fill_between(quantiles, 
                 np.array(qtes) - 1.96 * np.array(qte_ses),
                 np.array(qtes) + 1.96 * np.array(qte_ses),
                 alpha=0.3, label='95% CI')
plt.plot(quantiles, qtes, 'o-', markersize=8, linewidth=2, label='QTE')
plt.axhline(y=result.ate, color='red', linestyle='--', label=f"ATE = {result.ate:.3f}")
plt.axhline(y=true_ate, color='green', linestyle=':', label=f'True ATE = {true_ate}')

plt.xlabel('Quantile', fontsize=12)
plt.ylabel('Treatment Effect', fontsize=12)
plt.title('Panel Quantile Treatment Effects', fontsize=14)
plt.legend()
plt.xticks([0.1, 0.25, 0.5, 0.75, 0.9])
plt.tight_layout()
plt.show()

print("\n→ Relatively flat band suggests homogeneous effects across distribution")

### Unconditional QTE (Baseline)

For comparison, unconditional QTE without covariate adjustment:

In [ ]:
# Unconditional QTE (no covariates, just D=1 vs D=0)
uqte = panel_unconditional_qte(
    panel,
    quantile=0.5,
    n_bootstrap=500,
    cluster_bootstrap=True,
)

print("=== Unconditional QTE (Median) ===")
print(f"QTE: {uqte.qte:.3f}")
print(f"SE (cluster bootstrap): {uqte.qte_se:.3f}")
print(f"\nRIF-based QTE: {qte_50.qte:.3f}")
print("→ RIF adjusts for covariates; unconditional does not")

---

## Part 5: When to Use What?

| Method | Treatment | Estimand | Use Case |
|--------|-----------|----------|----------|
| **DML-CRE** | Binary | ATE | Standard treatment/control, mean effect |
| **DML-CRE Continuous** | Continuous | Marginal effect | Dose-response, intensity effects |
| **Panel RIF-QTE** | Binary | Unconditional QTE | Distributional effects (e.g., effects on low vs high earners) |
| **Panel Unconditional QTE** | Binary | Unconditional QTE | Quick baseline, no covariate adjustment |

### Decision Tree

```
Is treatment binary or continuous?
├── Binary
│   ├── Want mean effect? → DML-CRE
│   └── Want distributional effects? → Panel RIF-QTE
└── Continuous
    └── Want marginal effect? → DML-CRE Continuous
```

---

## Part 6: Practical Considerations

### 1. Minimum Sample Size

- **n_folds ≤ n_units**: Need enough units per fold for stable estimation
- **Rule of thumb**: At least 5-10 units per fold (so 50+ units for 5-fold CV)

In [ ]:
# Small sample example
data_small = generate_panel_dgp(n_units=30, n_periods=5, seed=999)
result_small = dml_cre(data_small['panel'], n_folds=3)  # Fewer folds for small N

print("=== Small Sample (30 units) ===")
print(f"ATE: {result_small.ate:.3f} (SE: {result_small.ate_se:.3f})")
print(f"True: {data_small['true_ate']}")
print("→ Wider CI with fewer units")

### 2. Balanced vs Unbalanced Panels

In [ ]:
# Create unbalanced panel (some units have fewer periods)
rng = np.random.default_rng(42)
n_units = 50

outcomes_list = []
treatment_list = []
covariates_list = []
unit_id_list = []
time_list = []

for i in range(n_units):
    # Random number of periods per unit (3-10)
    T_i = rng.integers(3, 11)
    for t in range(T_i):
        X = rng.normal(0, 1, 2)
        D = rng.binomial(1, 0.5)
        Y = 0.5 * i / n_units + 0.3 * X[0] + 2.0 * D + rng.normal(0, 1)
        
        outcomes_list.append(Y)
        treatment_list.append(D)
        covariates_list.append(X)
        unit_id_list.append(i)
        time_list.append(t)

panel_unbal = PanelData(
    outcomes=np.array(outcomes_list),
    treatment=np.array(treatment_list),
    covariates=np.array(covariates_list),
    unit_id=np.array(unit_id_list),
    time=np.array(time_list),
)

print(f"Balanced: {panel_unbal.is_balanced}")
print(f"Observations per unit: min={3}, max={10}")

result_unbal = dml_cre(panel_unbal, n_folds=5)
print(f"\nATE: {result_unbal.ate:.3f} (SE: {result_unbal.ate_se:.3f})")
print("→ DML-CRE handles unbalanced panels automatically")

### 3. Model Choice

In [ ]:
# Compare different model specifications
models = [
    ('linear', 'logistic'),
    ('ridge', 'logistic'),
    ('random_forest', 'random_forest'),
]

print("=== Model Comparison ===")
print(f"{'Outcome':>15} {'Propensity':>15} {'ATE':>10} {'SE':>10}")
print("-" * 55)

for outcome_model, propensity_model in models:
    result_m = dml_cre(panel, n_folds=5, 
                       outcome_model=outcome_model, 
                       propensity_model=propensity_model)
    print(f"{outcome_model:>15} {propensity_model:>15} {result_m.ate:>10.3f} {result_m.ate_se:>10.3f}")

print(f"\nTrue ATE: {true_ate}")
print("→ Results fairly robust to model choice for this DGP")

---

## Summary

### Key Takeaways

1. **Mundlak Projection** handles correlated unit effects by including $\bar{X}_i$
2. **Stratified K-Fold** by unit preserves panel structure in cross-fitting
3. **Clustered SE** accounts for within-unit correlation
4. **RIF-QTE** reveals treatment effects across the outcome distribution

### References

1. **Mundlak (1978)** - "On the Pooling of Time Series and Cross Section Data"
2. **Chernozhukov et al. (2018)** - "Double/Debiased Machine Learning"
3. **Firpo, Fortin, Lemieux (2009)** - "Unconditional Quantile Regressions"

---

*Created: Session 126 - Causal Inference Mastery*